In [0]:
from pyspark.sql.functions import lit

# Create DataFrames with specified sizes
df1 = spark.range(0, 2*100 * 1024 * 1024 // 8).withColumn("partition_id", lit(1))  # 30MB, assuming 8 bytes per row
df2 = spark.range(0, 2*30 * 1024 * 1024 // 8).withColumn("partition_id", lit(2))   # 3MB
df3 = spark.range(0, 2*30 * 1024 * 1024 // 8).withColumn("partition_id", lit(3))   # 3MB

# Union all partitions
df_union = df1.union(df2).union(df3)

# Repartition into 3 partitions
df_union = df_union.repartition(3, "partition_id")

display(df_union)

id,partition_id
0,2
1,2
2,2
3,2
4,2
5,2
6,2
7,2
8,2
9,2


In [0]:
from pyspark.sql.functions import spark_partition_id

df_partition_counts = df_union.groupBy("partition_id").count()
display(df_partition_counts)

partition_id,count
2,7864320
3,7864320
1,26214400


In [0]:
df_self_join = df_union.alias("left").join(
    df_union.alias("right"),
    on=["partition_id"],
    how="inner"
)

display(df_self_join)

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6025796404732355>, line 1
----> 1 spark.conf.set("spark.sql.adaptive.enabled", False)
      2 spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
      3 spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:51, in RuntimeConf.set(self, key, value)
     49 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     50 operation = proto.ConfigRequest.Operation(set=op_set)
---> 51 result = self._client.config(operation)
     52 for warn in result.warnings:
     53     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2193, in SparkConnectClient.config(self, operation)
   2191     raise SparkConnectException("Invalid state during retry exce

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", True)
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "20MB")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes","30MB")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6025796404732354>, line 1
----> 1 spark.conf.set("spark.sql.adaptive.enabled", True)
      2 spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", True)
      3 spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "20MB")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:51, in RuntimeConf.set(self, key, value)
     49 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     50 operation = proto.ConfigRequest.Operation(set=op_set)
---> 51 result = self._client.config(operation)
     52 for warn in result.warnings:
     53     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2193, in SparkConnectClient.config(self, operation)
   2191     raise SparkConnectException("Invalid state duri

In [0]:
df_union.write.format("delta").mode("overwrite").partitionBy("partition_id").saveAsTable("source_partitioned")

In [0]:
query = """
SELECT id, partition_id
FROM source_partitioned
WHERE partition_id = 2 and id = 100
"""

df_key = spark.sql(query)
display(df_key)

id,partition_id
100,2
